# F1 Top 10 Prediction

This notebook summarizes the reproducible machine learning pipeline used to predict whether a Formula 1 driver finishes in the top 10.

## Pipeline Commands

The project is primarily script-driven:

```powershell
python scripts/run_pipeline.py
python scripts/run_pipeline.py --with-fastf1
python scripts/evaluate_models.py
python scripts/predict_top10.py
python scripts/build_submission.py
```

In [1]:
from pathlib import Path
import json
import pandas as pd

DATA_PATH = Path('../data/final/f1_top10_model_dataset.csv')
METRICS_PATH = Path('../outputs/metrics.json')
COMPARISON_PATH = Path('../outputs/model_comparison.csv')
ROLLING_PATH = Path('../outputs/rolling_backtest.csv')

In [2]:
df = pd.read_csv(DATA_PATH)
df.shape

(6521, 166)

In [3]:
df.groupby('season').size()

season
2011    454
2012    478
2013    418
2014    407
2015    378
2016    462
2017    400
2018    420
2019    420
2020    340
2021    440
2022    440
2023    440
2024    479
2025    479
2026     66
dtype: int64

In [4]:
df['top10_finish'].value_counts(normalize=True).rename('share')

top10_finish
0    0.520012
1    0.479988
Name: share, dtype: float64

In [5]:
with METRICS_PATH.open(encoding='utf-8') as file:
    metrics = json.load(file)
metrics

{'model': 'random_forest',
 'train_rows': 5976,
 'test_rows': 479,
 'test_season': 2025,
 'accuracy': 0.7703549060542797,
 'precision': 0.76,
 'recall': 0.7916666666666666,
 'f1': 0.7755102040816326,
 'roc_auc': 0.844979079497908,
 'race_precision_at_10': 0.7666666666666666,
 'exact_top10_set_rate': 0.041666666666666664}

In [6]:
comparison = pd.read_csv(COMPARISON_PATH)
comparison.sort_values(['race_precision_at_10', 'f1'], ascending=False)

,model,train_rows,test_rows,train_start_season,train_end_season,test_season,accuracy,precision,recall,f1,roc_auc,race_precision_at_10,exact_top10_set_rate
0,hist_gradient_boosting,5976,479,2011,2024,2025,0.770355,0.777778,0.758333,0.767932,0.835722,0.775000,0.000000
1,random_forest,5976,479,2011,2024,2025,0.770355,0.760000,0.791667,0.775510,0.844979,0.766667,0.041667
2,extra_trees,5976,479,2011,2024,2025,0.759916,0.745098,0.791667,0.767677,0.839801,0.766667,0.041667
3,neural_network_mlp,5976,479,2011,2024,2025,0.716075,0.665605,0.870833,0.754513,0.817050,0.750000,0.041667
4,logistic_regression,5976,479,2011,2024,2025,0.736952,0.856250,0.570833,0.685000,0.808316,0.750000,0.041667


In [7]:
rolling = pd.read_csv(ROLLING_PATH)
rolling.groupby('model')[['accuracy', 'f1', 'roc_auc', 'race_precision_at_10']].mean().sort_values('race_precision_at_10', ascending=False)

,accuracy,f1,roc_auc,race_precision_at_10
model,,,,
random_forest,0.782046,0.780953,0.847065,0.775470
extra_trees,0.774859,0.776047,0.841025,0.770125
neural_network_mlp,0.770888,0.767813,0.838491,0.767034
hist_gradient_boosting,0.763550,0.758003,0.834553,0.761294
logistic_regression,0.748292,0.738627,0.820694,0.744724


## Notes

- The validation strategy is temporal, not random.
- Leakage columns such as final race position and race points are excluded during training.
- FastF1 enrichment is optional and currently focused on recent seasons.
- The neural network MLP is included as a baseline, but tree-based methods are currently more stable on this tabular dataset.